# 1.dfの作成

In [2]:
import pandas as pd

input = "../train/result0.xlsx"
df_all = pd.read_excel(input)

X_cols_SL = ["F14","F15","F16","F20","F0","R14","R15","R16","R20","R0"]
x_cols_hammett = ["Index 14","Index 15","Index 16","Index 20", "Index 0"]
y_cols_m = ["mwave1", "msrength1"]
y_cols_d = ["dwave1", "dstrength1"]

In [3]:
import numpy as np
import pandas as pd

def compute_scaled_distance_table(df, y_cols, target):
    """
    df:       pandas DataFrame
    y_cols:   目的変数カラム名のリスト（例: ['wavelength','osc_strength']）
    target:   numpy.array([530.64, 0.005])
    """

    Y = df[y_cols].values  # shape=(n_samples, 2)

    #--- スケール & オフセットを指定 ---
    # 波長  : 500 -> 530.64 を 1 に揃える => scale[0] = 530.64 - 500 = 30.64
    # 強度  :     0 -> 0.005  を 1 に揃える => scale[1] = 0.005
    scales = np.array([target[0] - 500.0,  target[1]])
    offsets = np.array([500.0,  0.0])

    #--- 変換後，理想値 (target) が [1,1] になるように ---
    Y_trans = (Y - offsets) / scales

    #--- [1,1] との差分 L2 ノルム を距離として採用 ---
    distances = np.linalg.norm(Y_trans - 1.0, axis=1)

    # 結果を DataFrame にまとめて返す
    return pd.DataFrame(
        {'distance_scaled': distances},
        index=df.index
    )

#――――――――――――――――――――――
# 使い方例
target = np.array([530.64, 0.005])
df_dist = compute_scaled_distance_table(df_all, y_cols_d, target)

# 小さい順に並べて出力
df_dist_sorted = df_dist.sort_values('distance_scaled')
df_dist_sorted.to_csv("../BO_result/distances/scaled_offset_distance_table.csv")


# 2.FPの作成

In [4]:
import pandas as pd
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

def create_fp_dataframe(df, smiles_col="dimer_smi", radius=2, n_bits=2048):
    """
    DataFrame から指定した SMILES 列を読み込み、Morgan フィンガープリント
    をビットベクトル化して新しい DataFrame として返す。

    Args:
        df (pd.DataFrame): 元の DataFrame（インデックスはそのまま利用）
        smiles_col (str): SMILES が入っているカラム名
        radius (int): Morgan fingerprint の半径
        n_bits (int): フィンガープリント長

    Returns:
        pd.DataFrame: 行数は df と同じ、列は FP_0～FP_{n_bits-1}
    """
    # 1. SMILES リスト取得
    smiles_list = df[smiles_col].tolist()
    
    # 2. フィンガープリント生成
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            fps.append(np.zeros(n_bits, dtype=int))
            continue
        bv = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
        arr = np.zeros((n_bits,), dtype=int)
        DataStructs.ConvertToNumpyArray(bv, arr)
        fps.append(arr)
    
    # 3. ndarray 化＆DataFrame 化
    fps_array = np.vstack(fps)  # shape = (len(df), n_bits)
    col_names = [f"FP_{i}" for i in range(n_bits)]
    df_fp = pd.DataFrame(fps_array, columns=col_names, index=df.index)
    
    return df_fp

# 使い方例

df_fp = create_fp_dataframe(df_all, smiles_col="dimer_smi", radius=2, n_bits=2048)
print(df_fp.shape)
df_fp.head()

# csv化
output_fp = "fp.csv"
df_fp.to_csv(output_fp, index=False)


[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerator
[15:54:46] DEPRECATION WARNING: please use MorganGenerat

(256, 2048)


In [5]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

def visualize_fp_substructures(df, smiles_col="dimer_smi",
                               radius=2, n_bits=2048, num_bits=8,
                               mols_per_row=4, subimg_size=(200,200)):
    """
    df の最初の SMILES について、フィンガープリントの
    上位 num_bits ビットに対応する原子環境をハイライト表示。
    PIL Image を直接表示します。
    """
    # 1) 分子取得
    smi = df.iloc[100][smiles_col]
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        raise ValueError("Invalid SMILES in first row")

    # 2) フィンガープリントと bitInfo
    bitInfo = {}
    _ = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits, bitInfo=bitInfo)

    # 3) 上位 num_bits オンビット
    on_bits = list(bitInfo.keys())[:num_bits]

    # 4) ハイライト原子リスト作成
    highlight_lists = []
    legends = []
    for bit in on_bits:
        envs = bitInfo[bit]
        if not envs:
            highlight_lists.append([])
            legends.append(f"bit {bit}: none")
        else:
            atom_idx, rad = envs[0]
            env = Chem.FindAtomEnvironmentOfRadiusN(mol, rad, atom_idx)
            amap = {}
            _ = Chem.PathToSubmol(mol, env, atomMap=amap)
            highlight_lists.append(list(amap.keys()))
            legends.append(f"bit {bit}")

    # 5) PIL Image を生成
    pil_img = Draw.MolsToGridImage(
        [mol] * len(highlight_lists),
        molsPerRow=mols_per_row,
        subImgSize=subimg_size,
        highlightAtomLists=highlight_lists,
        legends=legends
    )

    # 6) 直接表示
    display(pil_img)

# 使用例
# visualize_fp_substructures(df_all, smiles_col="dimer_smi", radius=2, n_bits=2048, num_bits=100)


In [4]:
df_fp = pd.read_csv("fp.csv")
df_all_FP = pd.concat([df_all, df_fp], axis=1)
# df_all_FP

# 3.小規模探索検証

In [9]:
%run -i "functions.py"

In [40]:
import numpy as np
import pandas as pd
from tqdm import trange
import warnings

warnings.filterwarnings("ignore")


def run_bo_trials(
    X_all: np.ndarray,
    Y_all: np.ndarray,
    hit_threshold: float,
    target_scaled: np.ndarray,
    offsets: np.ndarray,
    scales: np.ndarray,
    methods,
    init_size: int,
    trials: int,
    p: float,
    budget: int,
    eps: float,
):
    """
    BO の試行を走らせて生データを返す。
    """
    N, n_obj = Y_all.shape
    # 全点のスケーリング距離
    Y_scaled_all = (Y_all - offsets) / scales
    dist_all     = np.linalg.norm(Y_scaled_all - target_scaled, axis=1)
    # 生データ格納用
    hit_hist       = {m: np.zeros((trials, budget))         for m in methods}
    regret_hist    = {m: np.zeros((trials, budget, n_obj))  for m in methods}
    picks_idx      = {m: np.zeros((trials, budget), dtype=int) for m in methods}
    picks_Y        = {m: np.zeros((trials, budget, n_obj))  for m in methods}
    acq_hist       = {m: np.zeros((trials, budget))         for m in methods}
    acq_mean_hist  = {m: np.zeros((trials, budget))         for m in methods}

    for run in trange(trials, desc="BO Trials"):
        rng  = np.random.RandomState(run + 1)
        perm = rng.permutation(N)
        train_base = perm[:init_size].tolist()
        cand_base  = perm[init_size:].tolist()

        for m in methods:
            train_idx = train_base.copy()
            cand_idx  = cand_base.copy()
            cum_hits  = sum(dist_all[i] < hit_threshold for i in train_idx)
            cum_reg   = np.zeros(n_obj)

            # 初期モデル
            if m in ('lcb', 'lcb_per_dim', 'lcb_euclid', 'ei'):
                Y_tr_s = (Y_all[train_idx] - offsets) / scales
                models = fit_multi_gpy(X_all[train_idx], Y_tr_s)

            for it in range(budget):
                X_cd = X_all[cand_idx]
                if m == 'lcb_per_dim':
                    mu, var = predict_multi_gpy(models, X_cd)
                    scores  = l2_lcb_per_dim(mu, var, target_scaled, p, eps)
                elif m == 'lcb_euclid':
                    mu, var = predict_multi_gpy(models, X_cd)
                    scores  = l2_lcb_per_dim_euclid(mu, var, target_scaled, p, eps)
                elif m == 'lcb':
                    mu, var = predict_multi_gpy(models, X_cd)
                    scores  = l2_lcb_exact(mu, var, target_scaled, p, eps)
                elif m == 'ei':
                    mu, var = predict_multi_gpy(models, X_cd)
                    Y_tr_s  = (Y_all[train_idx] - offsets) / scales
                    y_min   = np.min(np.linalg.norm(Y_tr_s - target_scaled, axis=1))
                    scores  = l2_ei(mu, var, 1.0, y_min, eps)
                else:  # rand
                    scores = None

                # pick
                if scores is not None:
                    pick = np.argmax(scores)
                    acq_hist[m][run, it]      = scores[pick]
                    acq_mean_hist[m][run, it] = np.mean(scores)
                else:
                    pick = rng.randint(len(cand_idx))

                idx_pick = cand_idx.pop(pick)
                train_idx.append(idx_pick)

                # update metrics
                if dist_all[idx_pick] < hit_threshold:
                    cum_hits += 1
                hit_hist[m][run, it]    = cum_hits

                y_new = Y_all[idx_pick]
                y_new_scaled = (y_new - offsets) / scales
                cum_reg += np.abs(y_new_scaled - target_scaled)
                regret_hist[m][run, it] = cum_reg

                picks_idx[m][run, it] = idx_pick
                picks_Y[m][run, it]   = y_new

                # モデル更新
                if m in ('lcb','lcb_per_dim', 'lcb_euclid', 'ei'):
                    X_tr = X_all[train_idx]
                    Y_tr_s = (Y_all[train_idx] - offsets) / scales
                    for mdl in models:
                        mdl.set_XY(X_tr, Y_tr_s[:, [models.index(mdl)]])
                        mdl.optimize(messages=False, max_iters=5)

    return hit_hist, regret_hist, picks_idx, picks_Y, acq_hist, acq_mean_hist

def aggregate_all_bo_results(
    hit_hist,
    regret_hist,
    picks_idx,
    picks_Y,
    acq_hist,
    acq_mean_hist,
    methods,
    Y_cols,
    trials,
    budget,
):
    runs = [f"run{r+1}" for r in range(trials)]
    iters = [f"iter{t}" for t in range(1, budget+1)]

    # 1) hits
    df_hits_all = {
        m: pd.DataFrame(hit_hist[m], index=runs, columns=iters)
        for m in methods
    }

    # 2) regret: method → objective → DataFrame
    df_regret_all = {}
    for m in methods:
        df_regret_all[m] = {}
        for i, col in enumerate(Y_cols):
            arr = regret_hist[m][:, :, i]  # shape=(trials, budget)
            df_regret_all[m][col] = pd.DataFrame(arr, index=runs, columns=iters)

    # 3) picks index
    df_picks_all = {
        m: pd.DataFrame(picks_idx[m], index=runs, columns=iters)
        for m in methods
    }

    # 4) picks Y: method → objective → DataFrame
    df_Y_all = {}
    for m in methods:
        df_Y_all[m] = {}
        for i, col in enumerate(Y_cols):
            arr = picks_Y[m][:, :, i]
            df_Y_all[m][col] = pd.DataFrame(arr, index=runs, columns=iters)

    # 5) acquisition values
    df_acq_all = {
        m: pd.DataFrame(acq_hist[m], index=runs, columns=iters)
        for m in methods
    }
    df_acq_mean_all = {
        m: pd.DataFrame(acq_mean_hist[m], index=runs, columns=iters)
        for m in methods
    }

    return df_hits_all, df_regret_all, df_picks_all, df_Y_all, df_acq_all, df_acq_mean_all



def save_bo_results_all(
    file_initial: str,
    df_hits_all,
    df_regret_all,
    df_picks_all,
    df_Y_all,
    df_acq_all,
    df_acq_mean_all,
):
    # hits
    for m, df in df_hits_all.items():
        df.to_csv(f"{file_initial}_hits_{m}.csv")

    # regret（目的関数ごと）
    for m, obj_dfs in df_regret_all.items():
        for col, df in obj_dfs.items():
            df.to_csv(f"{file_initial}_regret_{m}_{col}.csv")

    # picks index
    for m, df in df_picks_all.items():
        df.to_csv(f"{file_initial}_picks_{m}.csv")

    # picks Y（目的関数ごと）
    for m, obj_dfs in df_Y_all.items():
        for col, df in obj_dfs.items():
            df.to_csv(f"{file_initial}_Y_{m}_{col}.csv")

    # acquisition
    for m, df in df_acq_all.items():
        df.to_csv(f"{file_initial}_acq_{m}.csv")

    # acquisition mean
    for m, df in df_acq_mean_all.items():
        df.to_csv(f"{file_initial}_acq_mean_{m}.csv")


In [18]:
# ユニーク値の種類数が「1より大きい」カラムだけを残す
df_FP_NONE1 = df_fp.loc[:, df_fp.nunique(dropna=False) > 1]

# カラム名リスト
X_cols_FP = df_FP_NONE1.columns.tolist()
print(f"FP columns with more than one unique value: {len(X_cols_FP)}")


FP columns with more than one unique value: 189


# FPでのBO


In [41]:

target = np.array([530.64 , 0.005])  # 目標ベクトルを適宜設定
df_hits, df_regret, df_picks, df_Y, df_acq, df_acq_mean= compare_all_methods(
    df_all_FP, X_cols_FP_NONE0 , y_cols_d, target,hit_threshold=0.15,
    methods=('lcb','rand'),
    init_size=5, trials=1, p=0.95, budget=None
)

[3.064e+01 5.000e-03]


Trials: 100%|██████████| 1/1 [00:16<00:00, 16.83s/it]


In [43]:
# 3) DataFrame に集約
df_hits, df_regret, df_picks, df_Y, df_acq, df_acq_mean = aggregate_all_bo_results(
    hit_hist, regret_hist, picks_idx, picks_Y,
    acq_hist, acq_mean_hist,
    methods, y_cols_d, trials, budget
)

In [57]:
# 4) CSV 出力
save_bo_results_all("../BO_result/target_relative/initsize5_all_methods",
    df_hits, df_regret, df_picks, df_Y, df_acq, df_acq_mean
)

# HammettでのBO

# Hammett + FP

In [37]:
X_cols_FP_Hammett = X_cols_FP + x_cols_hammett
print(f"Combined FP and Hammett columns: {len(X_cols_FP_Hammett)}")
FP_Hammett_scaler = StandardScaler()
scaled_X_FP_Hammett = FP_Hammett_scaler.fit_transform(df_all_FP[X_cols_FP_Hammett])
df_all_scaled = pd.concat([df_all_FP[y_cols_d], pd.DataFrame(scaled_X_FP_Hammett, columns=X_cols_FP_Hammett)], axis=1)
df_all_scaled 


Combined FP and Hammett columns: 194


,dwave1,dstrength1,FP_24,FP_33,FP_81,FP_84,FP_94,FP_115,FP_133,FP_138,...,FP_1983,FP_1996,FP_1999,FP_2037,FP_2042,Index 14,Index 15,Index 16,Index 20,Index 0
0,543.099536,0.0027,-0.125988,-0.125988,-0.625543,-1.133893,-0.480384,-0.125988,-0.221766,-0.221766,...,-0.221766,-0.125988,-0.258199,-0.625543,-0.221766,-0.205959,-0.617254,-0.205959,-0.205959,-0.205959
1,557.783845,0.0023,-0.125988,-0.125988,-0.625543,-1.133893,-0.480384,-0.125988,-0.221766,-0.221766,...,-0.221766,-0.125988,-0.258199,-0.625543,-0.221766,-0.205959,-0.617254,-0.205959,-0.617876,-0.617876
2,563.231695,0.0022,-0.125988,-0.125988,-0.625543,-1.133893,-0.480384,-0.125988,-0.221766,-0.221766,...,-0.221766,-0.125988,-0.258199,-0.625543,-0.221766,-0.205959,-0.617254,-0.205959,-0.860180,-0.860180
3,597.456597,0.0060,-0.125988,-0.125988,-0.625543,-1.133893,-0.480384,-0.125988,-0.221766,-0.221766,...,-0.221766,-0.125988,-0.258199,-0.625543,-0.221766,-0.205959,-0.617254,-0.205959,1.684014,1.684014
4,520.832569,0.0037,-0.125988,-0.125988,-0.625543,-1.133893,-0.480384,-0.125988,-0.221766,-0.221766,...,-0.221766,-0.125988,3.872983,-0.625543,-0.221766,-0.205959,-0.617254,-0.617876,-0.205959,-0.205959
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,549.088543,0.0099,-0.125988,-0.125988,1.598611,0.881917,-0.480384,-0.125988,-0.221766,-0.221766,...,-0.221766,-0.125988,-0.258199,1.598611,-0.221766,1.684014,1.689327,-0.860180,1.684014,1.684014
252,654.926803,0.0022,-0.125988,-0.125988,1.598611,0.881917,-0.480384,-0.125988,-0.221766,-0.221766,...,-0.221766,-0.125988,-0.258199,1.598611,-0.221766,1.684014,1.689327,1.684014,-0.205959,-0.205959
253,696.266597,0.0019,-0.125988,-0.125988,1.598611,0.881917,-0.480384,-0.125988,-0.221766,-0.221766,...,-0.221766,-0.125988,-0.258199,1.598611,-0.221766,1.684014,1.689327,1.684014,-0.617876,-0.617876
254,788.553031,0.0013,-0.125988,-0.125988,1.598611,0.881917,-0.480384,-0.125988,-0.221766,-0.221766,...,-0.221766,-0.125988,-0.258199,1.598611,-0.221766,1.684014,1.689327,1.684014,-0.860180,-0.860180


In [25]:
# 入力：pandas.DataFrame df_all, 説明変数カラム X_cols, 目的変数カラム Y_cols, 目標ベクトル target, 初期サイズ init_size

target = np.array([530.64 , 0.005])  # 目標ベクトルを適宜設定
df_hits, df_regret, df_picks, df_Y, df_acq = compare_all_methods(
    df_all_scaled, X_cols_FP_Hammett , y_cols_d, target,
    methods=('lcb','ei','rand'),
    init_size=5, trials=50, p=0.7, budget=251
)


Trials: 100%|██████████| 50/50 [38:01<00:00, 45.63s/it]


In [ ]:
# ファイルの保存
file_initial = "../BO_result/initsize5_LCB_per_dim_EI_05/FP+Hammett/FP+Hammett_p_0.7_init5"
df_hits.to_csv(f"{file_initial}_hits.csv")
df_regret_lcb = df_regret['lcb']
df_regret_ei = df_regret['ei']
df_regret_rand = df_regret['rand']
df_regret_lcb.to_csv(f"{file_initial}_regret_lcb.csv")
df_regret_ei.to_csv(f"{file_initial}_regret_ei.csv")
df_regret_rand.to_csv(f"{file_initial}_regret_rand.csv")

df_picks_lcb = df_picks['lcb']
df_picks_ei = df_picks['ei']
df_picks_rand = df_picks['rand']
df_picks_lcb.to_csv(f"{file_initial}_picks_lcb.csv")
df_picks_ei.to_csv(f"{file_initial}_picks_ei.csv")
df_picks_rand.to_csv(f"{file_initial}_picks_rand.csv")

df_Y_lcb = df_Y['lcb']
df_Y_ei = df_Y['ei']
df_Y_rand = df_Y['rand']
df_Y_lcb = pd.concat(df_Y_lcb, axis=1)
df_Y_ei = pd.concat(df_Y_ei, axis=1)
df_Y_rand = pd.concat(df_Y_rand, axis=1)
df_Y_lcb.to_csv(f"{file_initial}_Y_lcb.csv")
df_Y_ei.to_csv(f"{file_initial}_Y_ei.csv")
df_Y_rand.to_csv(f"{file_initial}_Y_rand.csv")

df_acq.to_csv(f"{file_initial}_acq.csv")